# 05 — Data Preprocessing & Cleaning

Cleans BlueSky and Lemmy data collected in notebooks 01 and 02.
Produces two cleaned tables in `moderation.db`:
- `bsky_posts_clean` — deduped, English-only, normalized BlueSky posts
- `lemmy_posts_clean` — combined title+body, normalized Lemmy posts

And one unified table ready for Perspective API scoring:
- `posts_for_scoring` — one row per post, `platform`, `post_id`, `text_clean`

**Cleaning steps applied:**
1. Remove null / empty text records
2. Remove very short texts (< 10 characters after stripping)
3. Deduplicate — keep one copy per unique text
4. Language filtering — keep English, flag others
5. Text normalization — strip URLs, @mentions, extra whitespace
6. Lemmy: combine title + body into single text field
7. Lemmy modlog: flag entries with no stated reason (keep, don't delete)
8. Export unified `posts_for_scoring` table

In [ ]:
import sys
sys.path.insert(0, "..")

import sqlite3
import re
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

DB_PATH = "../data/moderation.db"

def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def now_iso():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

print("✓ Setup complete")

## 1. Raw data audit — before cleaning

In [ ]:
conn = get_conn()

print("── RAW TABLE SIZES ──────────────────────────────")
for t in ["bsky_posts", "bsky_labels", "lemmy_posts", "lemmy_modlog"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<22}: {n:>6} rows")

print()
print("── BLUESKY ISSUES ───────────────────────────────")
issues = {
    "null/empty text"       : "SELECT COUNT(*) FROM bsky_posts WHERE text IS NULL OR TRIM(text) = ''",
    "text < 10 chars"       : "SELECT COUNT(*) FROM bsky_posts WHERE LENGTH(TRIM(COALESCE(text,''))) < 10",
    "duplicate texts"       : "SELECT COUNT(*) FROM (SELECT text FROM bsky_posts GROUP BY text HAVING COUNT(*) > 1)",
    "null lang"             : "SELECT COUNT(*) FROM bsky_posts WHERE lang IS NULL",
    "non-English lang"      : "SELECT COUNT(*) FROM bsky_posts WHERE lang IS NOT NULL AND lang NOT IN ('en','en-US','en-GB')",
}
for label, q in issues.items():
    print(f"  {label:<25}: {conn.execute(q).fetchone()[0]}")

print()
print("── LEMMY ISSUES ─────────────────────────────────")
lemmy_issues = {
    "null title"            : "SELECT COUNT(*) FROM lemmy_posts WHERE title IS NULL OR TRIM(title) = ''",
    "null body (OK)"        : "SELECT COUNT(*) FROM lemmy_posts WHERE body IS NULL OR TRIM(body) = ''",
    "modlog: no reason"     : "SELECT COUNT(*) FROM lemmy_modlog WHERE reason IS NULL OR TRIM(reason) = ''",
}
for label, q in lemmy_issues.items():
    print(f"  {label:<25}: {conn.execute(q).fetchone()[0]}")

conn.close()

## 2. Text normalization function

In [ ]:
_URL_RE      = re.compile(r'https?://\S+|www\.\S+')
_MENTION_RE  = re.compile(r'@[\w.\-]+(?:\.bsky\.social|@[\w.]+)?')
_WHITESPACE  = re.compile(r'\s+')

def normalize_text(text):
    """Strip URLs, @mentions, and collapse whitespace."""
    if not text or not isinstance(text, str):
        return ""
    text = _URL_RE.sub("", text)
    text = _MENTION_RE.sub("", text)
    text = _WHITESPACE.sub(" ", text).strip()
    return text

# Quick sanity check
samples = [
    "Check out https://example.com for more info @user.bsky.social!",
    "@alice.bsky.social @bob.bsky.social hello world",
    "  lots   of   whitespace   ",
]
for s in samples:
    print(f"  IN : {s}")
    print(f"  OUT: {normalize_text(s)}")
    print()

## 3. Language detection helper

In [ ]:
try:
    from langdetect import detect, LangDetectException
    LANGDETECT_AVAILABLE = True
    print("✓ langdetect available — will detect language for null-lang posts")
except ImportError:
    LANGDETECT_AVAILABLE = False
    print("⚠ langdetect not installed — install with: pip install langdetect")
    print("  Falling back to BlueSky lang field only")

ENGLISH_LANGS = {"en", "en-US", "en-GB", "en-AU", "en-CA"}

def detect_lang(text):
    """Return detected language code, or 'unknown'."""
    if not LANGDETECT_AVAILABLE or not text or len(text.strip()) < 20:
        return "unknown"
    try:
        return detect(text)
    except:
        return "unknown"

def is_english(lang_field, text):
    """
    Returns True if the post is English.
    Uses BlueSky lang field first; falls back to langdetect for null lang.
    """
    if lang_field in ENGLISH_LANGS:
        return True
    if lang_field is not None and lang_field not in ENGLISH_LANGS:
        return False
    # lang is null — try detection
    detected = detect_lang(text)
    return detected == "en"

## 4. Clean BlueSky posts

In [ ]:
conn = get_conn()
df_bsky = pd.read_sql_query("""
    SELECT uri, cid, author_did, author_handle, text, lang,
           post_created_at, like_count, reply_count, repost_count,
           search_query, fetched_at
    FROM bsky_posts
""", conn)
conn.close()

print(f"Loaded {len(df_bsky):,} raw BlueSky posts")
before = len(df_bsky)

In [ ]:
# ── Step 1: drop null / empty text ───────────────────────────────────────────
df_bsky = df_bsky[df_bsky["text"].notna() & (df_bsky["text"].str.strip() != "")]
print(f"After removing null/empty text : {len(df_bsky):,} rows")

# ── Step 2: normalize text ────────────────────────────────────────────────────
df_bsky["text_clean"] = df_bsky["text"].apply(normalize_text)

# ── Step 3: drop very short texts (after normalization) ───────────────────────
df_bsky = df_bsky[df_bsky["text_clean"].str.len() >= 10]
print(f"After removing texts < 10 chars: {len(df_bsky):,} rows")

# ── Step 4: deduplicate on normalized text ────────────────────────────────────
# Keep earliest post per unique normalized text
df_bsky = df_bsky.sort_values("post_created_at")
df_bsky["is_duplicate"] = df_bsky.duplicated(subset="text_clean", keep="first")
dupes_flagged = df_bsky["is_duplicate"].sum()
df_bsky = df_bsky[~df_bsky["is_duplicate"]].copy()
print(f"After removing {dupes_flagged} duplicates  : {len(df_bsky):,} rows")

# ── Step 5: language detection ────────────────────────────────────────────────
print("Detecting language (this may take a moment for null-lang posts)...")
df_bsky["is_english"] = df_bsky.apply(
    lambda r: is_english(r["lang"], r["text_clean"]), axis=1
)
non_english = (~df_bsky["is_english"]).sum()
print(f"Non-English posts flagged : {non_english}")

# Keep only English for Perspective API, but keep all in the clean table
df_bsky_english = df_bsky[df_bsky["is_english"]].copy()
print(f"English posts retained    : {len(df_bsky_english):,} rows")

# ── Step 6: add text length column ───────────────────────────────────────────
df_bsky["text_length"] = df_bsky["text_clean"].str.len()

print(f"\nSummary: {before:,} raw → {len(df_bsky):,} clean ({len(df_bsky_english):,} English)")

In [ ]:
# ── Save to DB ────────────────────────────────────────────────────────────────
conn = get_conn()
conn.execute("DROP TABLE IF EXISTS bsky_posts_clean")

df_bsky.drop(columns=["is_duplicate"]).to_sql(
    "bsky_posts_clean", conn, if_exists="replace", index=False
)

conn.commit()
conn.close()
print(f"✓ Saved {len(df_bsky):,} rows to bsky_posts_clean")

## 5. Clean Lemmy posts

In [ ]:
conn = get_conn()
df_lemmy = pd.read_sql_query("""
    SELECT p.post_id, p.instance, p.community_name, p.title, p.body,
           p.author_name, p.post_created_at, p.fetched_at,
           m.reason, m.removed, m.mod_name, m.actioned_at,
           CASE WHEN m.reason IS NULL OR TRIM(m.reason) = '' THEN 0 ELSE 1 END AS has_reason
    FROM lemmy_posts p
    LEFT JOIN lemmy_modlog m ON p.post_id = m.post_id AND p.instance = m.instance
""", conn)
conn.close()

print(f"Loaded {len(df_lemmy):,} Lemmy post+modlog records")
before_lemmy = len(df_lemmy)

In [ ]:
# ── Step 1: combine title + body ──────────────────────────────────────────────
# Title is always present; body is optional (link posts have no body)
df_lemmy["text_combined"] = df_lemmy.apply(
    lambda r: " ".join(filter(None, [
        str(r["title"]).strip() if pd.notna(r["title"]) else "",
        str(r["body"]).strip()  if pd.notna(r["body"])  else ""
    ])),
    axis=1
)

# ── Step 2: normalize text ────────────────────────────────────────────────────
df_lemmy["text_clean"] = df_lemmy["text_combined"].apply(normalize_text)

# ── Step 3: drop records with no usable text ──────────────────────────────────
df_lemmy = df_lemmy[df_lemmy["text_clean"].str.len() >= 10]
print(f"After removing short/empty texts: {len(df_lemmy):,} rows")

# ── Step 4: deduplicate ───────────────────────────────────────────────────────
df_lemmy = df_lemmy.sort_values("post_created_at")
dupes = df_lemmy.duplicated(subset="text_clean", keep="first").sum()
df_lemmy = df_lemmy[~df_lemmy.duplicated(subset="text_clean", keep="first")].copy()
print(f"After removing {dupes} duplicates      : {len(df_lemmy):,} rows")

# ── Step 5: add text length + no-reason flag ──────────────────────────────────
df_lemmy["text_length"] = df_lemmy["text_clean"].str.len()
df_lemmy["reason_missing"] = df_lemmy["has_reason"] == 0

print(f"\nLemmy modlog entries with no reason: {df_lemmy['reason_missing'].sum()} (flagged, kept)")
print(f"Summary: {before_lemmy:,} raw → {len(df_lemmy):,} clean")

In [ ]:
conn = get_conn()
conn.execute("DROP TABLE IF EXISTS lemmy_posts_clean")

df_lemmy.to_sql("lemmy_posts_clean", conn, if_exists="replace", index=False)

conn.commit()
conn.close()
print(f"✓ Saved {len(df_lemmy):,} rows to lemmy_posts_clean")

## 6. Build unified scoring table

One row per post across both platforms — this is the input for Perspective API in notebook 04.

In [ ]:
conn = get_conn()

# BlueSky — English only
df_score_bsky = pd.read_sql_query("""
    SELECT
        'bluesky'       AS platform,
        uri             AS post_id,
        text_clean,
        search_query    AS label_context,
        post_created_at AS created_at
    FROM bsky_posts_clean
    WHERE is_english = 1
""", conn)

# Lemmy — all records (Lemmy is mostly English across chosen instances)
df_score_lemmy = pd.read_sql_query("""
    SELECT
        'lemmy'         AS platform,
        post_id || '@' || instance AS post_id,
        text_clean,
        reason          AS label_context,
        post_created_at AS created_at
    FROM lemmy_posts_clean
    WHERE text_clean IS NOT NULL AND text_clean != ''
""", conn)

conn.close()

df_scoring = pd.concat([df_score_bsky, df_score_lemmy], ignore_index=True)
df_scoring["scored"] = 0  # flag for Perspective API (0 = not yet scored)

print(f"BlueSky posts ready for scoring : {len(df_score_bsky):,}")
print(f"Lemmy posts ready for scoring   : {len(df_score_lemmy):,}")
print(f"Total posts in scoring table    : {len(df_scoring):,}")

In [ ]:
conn = get_conn()
conn.execute("DROP TABLE IF EXISTS posts_for_scoring")

df_scoring.to_sql("posts_for_scoring", conn, if_exists="replace", index=False)

conn.commit()
conn.close()
print(f"✓ Saved {len(df_scoring):,} rows to posts_for_scoring")

## 7. Final summary — before vs after

In [ ]:
conn = get_conn()

summary = {
    "bsky_posts (raw)"       : conn.execute("SELECT COUNT(*) FROM bsky_posts").fetchone()[0],
    "bsky_posts_clean"       : conn.execute("SELECT COUNT(*) FROM bsky_posts_clean").fetchone()[0],
    "bsky_posts_clean (EN)"  : conn.execute("SELECT COUNT(*) FROM bsky_posts_clean WHERE is_english=1").fetchone()[0],
    "lemmy_posts (raw)"      : conn.execute("SELECT COUNT(*) FROM lemmy_posts").fetchone()[0],
    "lemmy_posts_clean"      : conn.execute("SELECT COUNT(*) FROM lemmy_posts_clean").fetchone()[0],
    "posts_for_scoring"      : conn.execute("SELECT COUNT(*) FROM posts_for_scoring").fetchone()[0],
}
conn.close()

print("── CLEANING SUMMARY ─────────────────────────────────────────")
for k, v in summary.items():
    print(f"  {k:<30}: {v:>6} rows")

## 8. Sample cleaned records

In [ ]:
conn = get_conn()

print("── BlueSky sample (cleaned) ─────────────────────────────────")
df_b = pd.read_sql_query("""
    SELECT SUBSTR(text_clean,1,100) AS text_clean, lang, is_english,
           text_length, search_query
    FROM bsky_posts_clean WHERE is_english=1 LIMIT 5
""", conn)
pd.set_option("display.max_colwidth", 105)
display(df_b)

print()
print("── Lemmy sample (cleaned) ───────────────────────────────────")
df_l = pd.read_sql_query("""
    SELECT instance, SUBSTR(text_clean,1,100) AS text_clean,
           reason, reason_missing, text_length
    FROM lemmy_posts_clean WHERE reason IS NOT NULL LIMIT 5
""", conn)
display(df_l)

conn.close()

## 9. Language distribution after cleaning

In [ ]:
conn = get_conn()
df_lang = pd.read_sql_query("""
    SELECT
        COALESCE(lang, 'unknown') AS lang,
        COUNT(*) AS count,
        is_english
    FROM bsky_posts_clean
    GROUP BY lang, is_english
    ORDER BY count DESC
""", conn)
conn.close()
print(df_lang.to_string(index=False))

## 10. Lemmy modlog — reason coverage by instance

In [ ]:
conn = get_conn()
df_reason = pd.read_sql_query("""
    SELECT
        instance,
        COUNT(*) AS total,
        SUM(CASE WHEN reason_missing = 0 THEN 1 ELSE 0 END) AS has_reason,
        SUM(CASE WHEN reason_missing = 1 THEN 1 ELSE 0 END) AS no_reason,
        ROUND(100.0 * SUM(CASE WHEN reason_missing = 0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_with_reason
    FROM lemmy_posts_clean
    GROUP BY instance
""", conn)
conn.close()
print(df_reason.to_string(index=False))